# BTC Analysis — publiczne API Binance
Bez klucza API. Dane z `data-api.binance.vision`.

In [2]:
import pandas as pd
import numpy as np
import requests
import plotly.graph_objects as go

BASE = 'https://data-api.binance.vision/api/v3'

def get_historical_klines(symbol='BTCUSDT', interval='15m', limit=96):
    resp = requests.get(f'{BASE}/klines',
        params={'symbol': symbol, 'interval': interval, 'limit': limit}, timeout=10)
    resp.raise_for_status()
    klines = resp.json()
    df = pd.DataFrame(klines, columns=[
        'timestamp','open','high','low','close','volume',
        'close_time','quote_asset_volume','number_of_trades',
        'taker_buy_base','taker_buy_quote','ignore'])
    df = df[['timestamp','open','high','low','close','volume','number_of_trades']]
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)
    return df.astype(float)

df = get_historical_klines()
print(f'Pobrano {len(df)} swiec')
df.head()

Pobrano 96 swiec


,open,high,low,close,volume,number_of_trades
timestamp,,,,,,
2026-05-27 17:00:00,75304.01,75344.01,74960.72,74968.25,264.12325,56385.0
2026-05-27 17:15:00,74968.26,75083.49,74880.00,74997.29,245.17889,47377.0
2026-05-27 17:30:00,74997.30,75036.78,74857.19,74908.00,292.46192,50841.0
2026-05-27 17:45:00,74908.00,74975.81,74685.58,74860.42,302.22813,58658.0
2026-05-27 18:00:00,74860.42,74938.01,74663.45,74738.01,161.96579,64637.0


In [3]:
def add_ema(df, periods=[20, 50, 200]):
    for p in periods:
        df[f'EMA_{p}'] = df['close'].ewm(span=p, adjust=False).mean()
    return df

def add_rsi(df, period=14):
    delta = df['close'].diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)
    avg_g = gain.ewm(com=period - 1, adjust=False).mean()
    avg_l = loss.ewm(com=period - 1, adjust=False).mean()
    rs    = avg_g / avg_l.replace(0, np.nan)
    df['RSI_14'] = (100 - 100 / (1 + rs)).round(2)
    return df

def add_zscore_spike(df, window=20, threshold=2.5):
    prices    = df['close'].values
    roll_mean = pd.Series(prices).rolling(window).mean().values
    roll_std  = pd.Series(prices).rolling(window).std().values
    z = np.where(roll_std > 0, (prices - roll_mean) / roll_std, 0)
    df['z_score']   = z.round(3)
    df['is_spike']  = np.abs(z) > threshold
    df['spike_dir'] = np.where(z >  threshold, 'UP',
                      np.where(z < -threshold, 'DOWN', ''))
    return df

df = add_ema(df)
df = add_rsi(df)
df = add_zscore_spike(df)
print(f'Wykryte skoki: {df["is_spike"].sum()}')
df[['close','EMA_20','EMA_50','RSI_14','z_score','spike_dir']].tail(10)

Wykryte skoki: 6


,close,EMA_20,EMA_50,RSI_14,z_score,spike_dir
timestamp,,,,,,
2026-05-28 14:30:00,72855.80,73239.344987,73488.748630,38.56,-1.975,
2026-05-28 14:45:00,73033.06,73219.698797,73470.878487,43.79,-1.107,
2026-05-28 15:00:00,72861.93,73185.625579,73446.998155,40.23,-1.610,
2026-05-28 15:15:00,72968.35,73164.932666,73428.227639,43.32,-1.068,
2026-05-28 15:30:00,73013.39,73150.500032,73411.959496,44.62,-0.804,
2026-05-28 15:45:00,72928.73,73129.379076,73393.009320,42.64,-1.032,
2026-05-28 16:00:00,73109.55,73127.490593,73381.893268,47.96,-0.262,
2026-05-28 16:15:00,73150.02,73129.636251,73372.800199,49.10,-0.058,
2026-05-28 16:30:00,73357.72,73151.358512,73372.208818,54.60,0.842,


In [4]:
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df['open'], high=df['high'],
    low=df['low'], close=df['close'], name='BTC/USDT'))
for col, color in [('EMA_20','rgba(0,255,0,0.7)'),('EMA_50','rgba(255,165,0,0.7)'),('EMA_200','rgba(255,0,0,0.8)')]:
    if col in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df[col], name=col, line=dict(color=color, width=1.2)))
ups   = df[df['spike_dir'] == 'UP']
downs = df[df['spike_dir'] == 'DOWN']
if len(ups):
    fig.add_trace(go.Scatter(x=ups.index, y=ups['high']*1.001, mode='markers',
        marker=dict(symbol='triangle-up', size=10, color='lime'), name='Skok UP'))
if len(downs):
    fig.add_trace(go.Scatter(x=downs.index, y=downs['low']*0.999, mode='markers',
        marker=dict(symbol='triangle-down', size=10, color='red'), name='Skok DOWN'))
fig.update_layout(title='BTC/USDT — swiece 15m + EMA + skoki z-score',
    yaxis_title='Cena (USDT)', template='plotly_dark', height=600,
    xaxis_rangeslider_visible=True)
fig.show()

In [5]:
print('=== Statystyki ===')
print(df['close'].describe().round(2))
print(f'ATH: {df["high"].max():.2f} USDT')
print(f'ATL: {df["low"].min():.2f} USDT')
print(f'Zmiana %: {((df["close"].iloc[-1] / df["close"].iloc[0]) - 1) * 100:.2f}%')
print(f'Skoki: {df["is_spike"].sum()}')
print(df[df['is_spike']][['close','z_score','spike_dir']])

=== Statystyki ===
count       96.00
mean     73860.74
std        777.47
min      72785.20
25%      73210.64
50%      73500.00
75%      74478.12
max      75309.33
Name: close, dtype: float64
ATH: 75415.81 USDT
ATL: 72582.82 USDT
Zmiana %: -2.21%
Skoki: 6
                        close  z_score spike_dir
timestamp                                       
2026-05-28 03:15:00  73746.57   -3.255      DOWN
2026-05-28 03:30:00  73430.01   -3.217      DOWN
2026-05-28 03:45:00  73258.01   -2.773      DOWN
2026-05-28 04:00:00  73082.08   -2.508      DOWN
2026-05-28 13:15:00  73160.52   -2.531      DOWN
2026-05-28 13:30:00  72785.20   -3.512      DOWN
